In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as widgets
from ipywidgets import interact

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import *
    
%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
# implement cells as star shaped polygons.



def generate_single_cell(size = 201, radius = 32, nuc_frac = 0.3, rough = 0.9, K = 20, elong = 1.6, angle_deg = 30, beta = 1.9):
    # Random fourrier coefficients for a star shaped polygon
    rng = np.random.default_rng(
        seed=42
        )
    a, b = rng.standard_normal(K), rng.standard_normal(K)
    k = np.arange(1, K + 1)
    
    # get polar coordinates of the star shaped polygon
    # generate coordinate grid with origin in the center
    y, x = np.mgrid[:size, :size] - (size-1)/2
    t = np.deg2rad(angle_deg)
    # rotate the coordinate grid by angle_deg
    xr, yr = x * np.cos(t) + y * np.sin(t), -x * np.sin(t) + y * np.cos(t)
    # elongation factor along the x-axis
    s = np.sqrt(elong)
    # get polar coordinates
    rho = np.hypot(xr/s, yr*s)
    phi = np.arctan2(yr*s, xr/s)
    
    def boundary(R, kappa, extra = 0.0):
        # boundry radius as function of angle adding wobbles
        # get more fat lobes 
        w = kappa* k ** -(beta + extra)
        # decouple the scale from the amount of wobble
        w = w / (np.linalg.norm(w) + 1e-12) * kappa
        # generate lumps at random locations (cosine and sine components)
        g = np.cos(phi[..., None]*k) @ (w*a) + np.sin(phi[..., None]*k) @ (w*b)
        # use exponential to get a definitively positive radius
        return R * np.exp(g - kappa**2)

    # get boundaries
    r_cell = boundary(radius, rough)
    r_nuc  = boundary(radius * nuc_frac, rough * 0.6, extra=1.5)

    # check if inside cell / bucleus
    cell = rho <= r_cell
    nuc  = rho <= np.minimum(r_nuc, r_cell - 1.5)     # leave a cytoplasmic rim

    return cell, nuc, rho, phi

cell, nuc, rho, phi = generate_single_cell()

# plt.imshow(rho)
# plt.show()
# plt.imshow(phi)
# plt.show()
# plt.imshow(cell)
# plt.show()
# plt.imshow(nuc)
# plt.show()
    
    

In [ ]:
def plot_interactive_cell(size, radius, nuc_frac, rough, K, elong, angle_deg, beta):
    # Generate the masks
    cell, nuc, rho, phi = single_cell(
        size=size, radius=radius, nuc_frac=nuc_frac, 
        rough=rough, K=K, elong=elong, 
        angle_deg=angle_deg, beta=beta
    )
    
    # Create an RGB image background (black)
    img = np.zeros((size, size, 3))
    
    # Assign colors using the masks
    # Cytoplasm (light green)
    img[cell] = [0.2, 0.8, 0.3]
    # Nucleus (light blue) overwrites the cytoplasm where it exists
    img[nuc] = [0.3, 0.5, 0.9]
    
    # Plotting
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Synthetic Cell Generation')
    plt.show()

# Set up the interactive sliders with reasonable bounds based on your defaults
interact(plot_interactive_cell,
         size=widgets.IntSlider(min=100, max=500, step=10, value=201, description='Size'),
         radius=widgets.FloatSlider(min=10.0, max=150.0, step=1.0, value=32.0, description='Radius'),
         nuc_frac=widgets.FloatSlider(min=0.1, max=0.9, step=0.05, value=0.3, description='Nuc Frac'),
         rough=widgets.FloatSlider(min=0.0, max=2.0, step=0.1, value=0.9, description='Roughness'),
         K=widgets.IntSlider(min=1, max=50, step=1, value=20, description='K (Harmonics)'),
         elong=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.6, description='Elongation'),
         angle_deg=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=30.0, description='Angle (deg)'),
         beta=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.9, description='Beta'));

# 2) Synthetic tissue generation